In [ ]:
# Roomscape: one-cell Google Colab setup and reconstruction
# Edit these values first. Use a short, complete walkthrough segment, not a random cut.
REPO_URL = "https://github.com/s2kumar2007/roomscape.git"
DRIVE_VIDEO = ""                 # Example: roomscape/room.mp4, relative to MyDrive
CLIP_START_SECONDS = 0            # Start of the useful walkthrough in the source video
CLIP_DURATION_SECONDS = 300        # 5 minutes; increase if needed to cover the full space
TARGET_FRAMES = 300
FRAMES = TARGET_FRAMES
TRIM_START = CLIP_START_SECONDS
TRIM_END = 0
# COLMAP configuration: use_gpu: False (via --no-gpu flag)
RUN_TESTS = False                  # Set True to run repository pytest before reconstruction
RESUME = False                     # Set True only to continue a previous interrupted run

import os
import shutil
import subprocess
import sys
from pathlib import Path

def run(command, **kwargs):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(command, check=True, **kwargs)

# Install system tools and Python packages.
if shutil.which("ffmpeg") is None or shutil.which("colmap") is None:
    run(["apt-get", "-qq", "update"])
    run(["apt-get", "-qq", "install", "-y", "ffmpeg", "colmap"])
run([sys.executable, "-m", "pip", "install", "-q",
     "nerfstudio", "numpy", "opencv-contrib-python", "pyyaml", "matplotlib", "pytest"])

# Clone/update the repository.
REPO_DIR = Path("/content/roomscape")
if (REPO_DIR / ".git").exists():
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    run(["git", "clone", REPO_URL, str(REPO_DIR)])

# Mount Drive and select the source video without copying the full 5 GB file.
from google.colab import drive
drive.mount("/content/drive")
if not DRIVE_VIDEO:
    DRIVE_VIDEO = input("Video path relative to MyDrive (example: roomscape/room.mp4): ").strip()
source_video = Path(DRIVE_VIDEO)
if not source_video.is_absolute():
    source_video = Path("/content/drive/MyDrive") / source_video
if not source_video.is_file():
    raise FileNotFoundError(f"Video not found: {source_video}")

if CLIP_START_SECONDS < 0 or CLIP_DURATION_SECONDS <= 0:
    raise ValueError("Clip start must be non-negative and duration must be positive.")

# Stream-copy only the chosen section into Colab local storage; this avoids copying
# the entire source video and preserves the original video quality.
LOCAL_INPUT = Path("/content/roomscape_input")
LOCAL_INPUT.mkdir(parents=True, exist_ok=True)
short_video = LOCAL_INPUT / "room_short.mp4"
run([
    "ffmpeg", "-y", "-hide_banner", "-loglevel", "warning",
    "-ss", str(CLIP_START_SECONDS), "-i", str(source_video),
    "-t", str(CLIP_DURATION_SECONDS), "-map", "0:v:0",
    "-c:v", "copy", "-an", "-movflags", "+faststart", str(short_video),
])
if not short_video.is_file() or short_video.stat().st_size == 0:
    raise RuntimeError("FFmpeg did not create the shortened video.")

# Show the GPU, then run the pipeline. --no-gpu selects CPU SIFT in COLMAP;
# Nerfstudio training still uses the Colab CUDA GPU.
subprocess.run(["nvidia-smi"], check=False)
os.environ["QT_QPA_PLATFORM"] = "offscreen"
os.chdir(REPO_DIR)

if RUN_TESTS:
    run([sys.executable, "-m", "pytest", "-q"])

output_dir = Path("/content/roomscape_output")
pipeline_command = [
    sys.executable, "pipeline/run_pipeline.py",
    "--video", str(short_video),
    "--out", str(output_dir),
    "--mode", "fast",
    "--frames", str(TARGET_FRAMES),
    "--no-gpu",
]
if RESUME:
    pipeline_command.append("--resume")
run(pipeline_command)

# Copy reports and the compact viewer scene back to Drive. Original video and
# extracted frames are deliberately excluded.
drive_results = Path("/content/drive/MyDrive/roomscape_results")
drive_results.mkdir(parents=True, exist_ok=True)
report_files = [
    "report.md", "report.json", "timings.json", "frames_report.json",
    "reconstruct_report.json", "transform.json", "scene_info.json",
    "collision.json", "occupancy.png", "export_report.json",
    "cameras_aligned.json",
]
for name in report_files:
    item = output_dir / name
    if item.is_file():
        shutil.copy2(item, drive_results / name)
for folder in ("scene", "debug"):
    item = output_dir / folder
    if item.is_dir():
        shutil.copytree(item, drive_results / folder, dirs_exist_ok=True)
viewer_scene = REPO_DIR / "viewer" / "public" / "scene"
if viewer_scene.is_dir():
    shutil.copytree(viewer_scene, drive_results / "viewer_scene", dirs_exist_ok=True)

print(f"Done. Reports and viewer files are in: {drive_results}")
print(f"Short clip used: {short_video} ({short_video.stat().st_size / 1e6:.1f} MB)")